# Notebook 5 — Evaluation and pitfalls

*Part of **MRI Reconstruction from k-space to Deep Learning: A Hands-on Introduction***

Notebook 4 ended with a clear winner: the unrolled network had a sixth of the zero-filled error and ran in milliseconds. Should a hospital use it tomorrow? This notebook asks the questions a careful reviewer, radiologist or regulator would ask before trusting a reconstruction method:

- Do our **error measures** agree with each other, and with what matters clinically?
- Does the method keep **small structures** it never saw during training?
- What happens when the **scan changes**: another mask, a higher acceleration, more noise?
- What happens on images that look **different from the training data**?

**Learning objectives.** By the end of this notebook you will be able to:

1. Compute NRMSE, PSNR and SSIM, and explain why they can rank methods differently.
2. Test whether a reconstruction keeps small features, and separate limits of the **sampling** from limits of the **method**.
3. Measure how reconstruction methods behave under **distribution shift**.
4. Use a checklist to evaluate claims about AI reconstruction methods critically.

**Prerequisites:** Notebooks 1–4. Nothing is trained here: the notebook downloads the networks trained in Notebook 4, so it runs in a few minutes on a CPU.

**Time:** about 60–75 minutes.

As always, answer every **🤔 Predict** question before running the cell.

## 1. Setup

The cells below load the libraries, the helper functions and compressed sensing from earlier notebooks, the random phantom generator, the same test phantoms as Notebook 4, and both trained networks. The unrolled network now takes the sampling mask as an input, so we can test it with masks it was not trained on.

In [ ]:
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import uniform_filter, gaussian_filter
from skimage.data import shepp_logan_phantom, camera
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

plt.rcParams["figure.dpi"] = 100
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # line colors, always used in this order
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}. This notebook does not train anything, so a CPU is fine.")

In [ ]:
# ---- From Notebooks 1-3 (now working on stacks of images too) ------------------
def fft2c(image):
    """Centered, orthonormal 2D FFT over the last two axes: image(s) -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def ifft2c(kspace):
    """Centered, orthonormal inverse 2D FFT over the last two axes: k-space -> image(s)."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def show(images, titles, cmap="gray", vmin=None, vmax=None):
    """Show images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    for ax, im, title in zip(np.atleast_1d(axes), images, titles):
        ax.set_title(title)
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def nrmse(recon, reference):
    """Normalized root-mean-square error: 0 is perfect, 1 is as large as the image itself."""
    return np.linalg.norm(recon - reference) / np.linalg.norm(reference)


def tidy(ax):
    """Quiet grid and axes for line plots."""
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)


def variable_density_lines(N, R, center_fraction=0.08, decay=3, seed=0):
    """N // R lines: a fully sampled center, the rest random and more likely near the center."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    distance = np.abs(candidates - N // 2) / (N / 2)
    probability = (1 - distance) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False,
                     p=probability / probability.sum())] = True
    return lines


# ---- Compressed sensing from Notebook 3, for comparison --------------------------
SQRT2 = np.sqrt(2)


def haar2(x, levels=4):
    c = x.astype(complex).copy()
    n = c.shape[0]
    for _ in range(levels):
        a = c[:n, :n]
        a = np.hstack([(a[:, 0::2] + a[:, 1::2]) / SQRT2, (a[:, 0::2] - a[:, 1::2]) / SQRT2])
        a = np.vstack([(a[0::2, :] + a[1::2, :]) / SQRT2, (a[0::2, :] - a[1::2, :]) / SQRT2])
        c[:n, :n] = a
        n //= 2
    return c


def ihaar2(c, levels=4):
    x = c.copy()
    n = x.shape[0] // 2 ** (levels - 1)
    for _ in range(levels):
        a = x[:n, :n]
        h = n // 2
        b = np.empty_like(a)
        b[0::2, :] = (a[:h, :] + a[h:, :]) / SQRT2
        b[1::2, :] = (a[:h, :] - a[h:, :]) / SQRT2
        a2 = np.empty_like(a)
        a2[:, 0::2] = (b[:, :h] + b[:, h:]) / SQRT2
        a2[:, 1::2] = (b[:, :h] - b[:, h:]) / SQRT2
        x[:n, :n] = a2
        n *= 2
    return x


def soft_threshold(c, lam):
    magnitude = np.abs(c)
    return c * np.maximum(magnitude - lam, 0) / np.maximum(magnitude, 1e-12)


def cs_reconstruct(kspace_measured, lines, lam=0.01, n_iter=200, levels=4):
    n = kspace_measured.shape[0]
    coarse = np.zeros((n, n), dtype=bool)
    coarse[: n // 2**levels, : n // 2**levels] = True
    x = ifft2c(kspace_measured)
    for _ in range(n_iter):
        k = np.where(lines[:, None], kspace_measured, fft2c(x))
        x = ifft2c(k)
        w = haar2(x, levels)
        x = ihaar2(np.where(coarse, w, soft_threshold(w, lam)), levels)
    return x

In [ ]:
def random_phantom(rng, N=128, oversample=2):
    """A random head-like phantom: a bright skull ring, a darker brain,
    4-8 internal structures and 1-3 small 'lesions'. Values between 0 and 1."""
    M = N * oversample                                  # draw at higher resolution, then average down
    y, x = np.mgrid[-1:1:M * 1j, -1:1:M * 1j]
    img = np.zeros((M, M))

    def ellipse(cx, cy, a, b, angle, value):
        c, s = np.cos(angle), np.sin(angle)
        xr = (x - cx) * c + (y - cy) * s
        yr = -(x - cx) * s + (y - cy) * c
        img[(xr / a) ** 2 + (yr / b) ** 2 <= 1] += value

    def random_point_inside(spread):
        r, phi = rng.uniform(0, spread), rng.uniform(0, 2 * np.pi)
        return cx + r * (a - t) * 0.8 * np.cos(phi), cy + r * (b - t) * 0.8 * np.sin(phi)

    # head outline: skull ring around a darker brain
    cx, cy = rng.uniform(-0.05, 0.05, 2)
    a, b = rng.uniform(0.6, 0.75), rng.uniform(0.8, 0.92)
    angle = rng.uniform(-0.2, 0.2)
    t = rng.uniform(0.04, 0.07)                         # skull thickness
    ellipse(cx, cy, a, b, angle, 1.0)
    ellipse(cx, cy, a - t, b - t, angle, rng.uniform(0.15, 0.3) - 1.0)

    # internal structures of different sizes and brightness
    for _ in range(rng.integers(4, 9)):
        ex, ey = random_point_inside(0.6)
        size = rng.uniform(0.04, 0.25)
        ellipse(ex, ey, size * rng.uniform(0.4, 1.0), size, rng.uniform(0, np.pi), rng.uniform(-0.2, 0.25))

    # a few small bright or dark "lesions"
    for _ in range(rng.integers(1, 4)):
        ex, ey = random_point_inside(0.7)
        size = rng.uniform(0.015, 0.04)
        ellipse(ex, ey, size, size * rng.uniform(0.6, 1.0), rng.uniform(0, np.pi),
                rng.uniform(0.1, 0.3) * rng.choice([-1, 1]))

    img = np.clip(img, 0, 1)
    return img.reshape(N, oversample, N, oversample).mean(axis=(1, 3))


# Exactly the same data as Notebook 4 (same random seed), so the test phantoms are the same.
N, R = 128, 4
N_TRAIN, N_VAL, N_TEST = 1000, 100, 100
rng = np.random.default_rng(0)
truth = np.stack([random_phantom(rng, N) for _ in range(N_TRAIN + N_VAL + N_TEST)])
test = slice(N_TRAIN + N_VAL, None)
lines = variable_density_lines(N, R)       # the mask both networks were trained with
print(f"{N_TEST} test phantoms, mask with {lines.sum()} of {N} lines (R = {R})")

In [ ]:
def conv_block(c_in, c_out):
    """Two 3x3 convolutions, each followed by ReLU."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """A small U-Net that predicts a correction to the zero-filled image."""

    def __init__(self, channels=2, base=16):
        super().__init__()
        # encoder: features at 128, 64 and 32 pixels
        self.enc1 = conv_block(channels, base)
        self.enc2 = conv_block(base, 2 * base)
        self.enc3 = conv_block(2 * base, 4 * base)
        # bottleneck at 16 pixels
        self.bottom = conv_block(4 * base, 8 * base)
        # decoder: upsample, join with the matching encoder features (skip connection), convolve
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * base, 4 * base, 2, stride=2), conv_block(8 * base, 4 * base)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * base, 2 * base, 2, stride=2), conv_block(4 * base, 2 * base)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * base, base, 2, stride=2), conv_block(2 * base, base)
        self.out = nn.Conv2d(base, channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        b = self.bottom(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return x + self.out(d1)          # residual: input + learned correction


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def to_tensor(c):
    """Complex images (B, N, N) -> real tensor (B, 2, N, N)."""
    return torch.tensor(np.stack([c.real, c.imag], axis=1), dtype=torch.float32)


def to_complex(t):
    t = t.detach().cpu().numpy()
    return t[:, 0] + 1j * t[:, 1]


def tfft2c(x):
    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def tifft2c(k):
    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def data_consistency(x, k_measured, mask):
    """As in Notebook 4, but the sampling mask is now an argument, so we can test other masks."""
    image = torch.complex(x[:, 0], x[:, 1])
    image = tifft2c(torch.where(mask, k_measured, tfft2c(image)))
    return torch.stack([image.real, image.imag], dim=1)


class SmallCNN(nn.Module):
    def __init__(self, channels=2, width=16, depth=5):
        super().__init__()
        layers = [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(width, channels, 3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class UnrolledNet(nn.Module):
    def __init__(self, cascades=4):
        super().__init__()
        self.cnns = nn.ModuleList([SmallCNN() for _ in range(cascades)])

    def forward(self, x, k_measured, mask):
        for cnn in self.cnns:
            x = data_consistency(cnn(x), k_measured, mask)
        return x


def load_pretrained(model, filename):
    """Download the weights trained in Notebook 4 from the module's GitHub repository."""
    url = f"https://raw.githubusercontent.com/anupbagale/mri-recon-tutorial/main/weights/{filename}"
    urllib.request.urlretrieve(url, filename)
    model.load_state_dict(torch.load(filename, map_location=device))
    return model.to(device).eval()


unet = load_pretrained(UNet(), "unet_r4.pt")
unrolled = load_pretrained(UnrolledNet(), "unrolled_r4.pt")
print("Loaded the U-Net and the unrolled network trained in Notebook 4.")

In [ ]:
METHODS = ["zero-filled", "compressed sensing", "U-Net", "unrolled network"]


def reconstruct_all(image, lines, noise=0.0, seed=0):
    """Simulate a scan of `image` with the given lines (and optional k-space noise),
    then reconstruct it with all four methods. Returns a dict of magnitude images."""
    k = fft2c(image)
    if noise > 0:
        noise_rng = np.random.default_rng(seed)
        k = k + noise * (noise_rng.standard_normal(k.shape) + 1j * noise_rng.standard_normal(k.shape)) / np.sqrt(2)
    measured = k * lines[:, None]
    zf = ifft2c(measured)
    x = to_tensor(zf[None]).to(device)
    k_t = torch.tensor(measured[None], dtype=torch.complex64).to(device)
    mask = torch.tensor(lines[:, None], device=device)
    with torch.no_grad():
        unet_out = np.abs(to_complex(unet(x)))[0]
        unrolled_out = np.abs(to_complex(unrolled(x, k_t, mask)))[0]
    return {"zero-filled": np.abs(zf),
            "compressed sensing": np.abs(cs_reconstruct(measured, lines)),
            "U-Net": unet_out,
            "unrolled network": unrolled_out}

## 2. Three ways to measure error

So far we have used **NRMSE**. Two other measures are standard in MRI reconstruction papers:

- **PSNR** (peak signal-to-noise ratio), in decibels: $\text{PSNR} = 10 \log_{10} \left( \text{peak}^2 / \text{MSE} \right)$. Higher is better. For a fixed image, it is just another way of writing the squared error, so it carries the same information as NRMSE.
- **SSIM** (structural similarity; Wang et al., 2004) compares *local* brightness, contrast and structure in small windows, and averages over the image. It is 1 for a perfect image. It was designed to match human perception better than squared error.

> 🤔 **Predict:** We make two damaged versions of a test phantom with **exactly the same NRMSE**: one blurred, one with added noise. Will PSNR and SSIM also agree that they are equally good?

In [ ]:
def psnr(recon, reference):
    """Peak signal-to-noise ratio in dB (higher is better). Our images range from 0 to 1."""
    return peak_signal_noise_ratio(reference, recon, data_range=1.0)


def ssim(recon, reference):
    """Structural similarity, from 0 to 1 (higher is better)."""
    return structural_similarity(reference, recon, data_range=1.0)

In [ ]:
t = truth[test][0]
blurred = gaussian_filter(t, 1.2)

# noise scaled so that the noisy image has exactly the same NRMSE as the blurred one
noise_pattern = np.random.default_rng(0).standard_normal(t.shape)
noisy = t + noise_pattern * nrmse(blurred, t) * np.linalg.norm(t) / np.linalg.norm(noise_pattern)

print(f"{'':<10}{'NRMSE':>8}{'PSNR':>9}{'SSIM':>8}")
for name, img in [("blurred", blurred), ("noisy", noisy)]:
    print(f"{name:<10}{nrmse(img, t):>8.3f}{psnr(img, t):>7.1f} dB{ssim(img, t):>8.3f}")
show([t, blurred, noisy], ["ground truth", "blurred", "noisy (same NRMSE)"], vmin=0, vmax=1)

NRMSE and PSNR cannot tell the two apart: they only measure the *total* squared error, not where it is or what it looks like. SSIM strongly prefers the blurred image, because it keeps the local structure of the flat regions.

Which one is "better"? It depends on the task. Noise can hide a faint lesion; blur can merge two small ones. **No single number captures diagnostic quality.**

## 3. Do the metrics agree on our four methods?

> 🤔 **Predict:** In Notebook 4, the U-Net had a lower NRMSE than compressed sensing, but kept much less of a small lesion's contrast. Which of the two will SSIM prefer?

In [ ]:
n_eval = 20
scores = {m: [] for m in METHODS}
for i in range(n_eval):
    t = truth[test][i]
    recons = reconstruct_all(t, lines)
    for m in METHODS:
        scores[m].append((nrmse(recons[m], t), psnr(recons[m], t), ssim(recons[m], t)))

print(f"{'method':<20}{'NRMSE ↓':>9}{'PSNR ↑':>11}{'SSIM ↑':>9}   (mean of {n_eval} test phantoms)")
for m in METHODS:
    e, p, s = np.mean(scores[m], axis=0)
    print(f"{m:<20}{e:>9.3f}{p:>8.1f} dB{s:>9.3f}")

**Interpretation:** NRMSE and PSNR always agree (one is computed from the other), and both rank the U-Net above compressed sensing. **SSIM reverses that order.** This matches what you measured in Notebook 4: the U-Net's smooth, blurred output has a low squared error, but loses local structure such as a small lesion's contrast, and SSIM notices.

The unrolled network wins on all three measures, so here the conclusion is safe. But when metrics disagree, a paper that reports only the one where its method wins is telling half the story.

## 4. Small features the networks never saw

Our training phantoms contained only **ellipses**. A real patient can have anything: a tiny calcification, a thin vessel, a surgical clip. We add three small features to a test phantom, none of them an ellipse:

- a **small bright square**, 4 × 4 pixels;
- a **thin vertical line**, 1 pixel wide;
- a **thin horizontal line**, 1 pixel wide.

> 🤔 **Predict:** Which of the three features will each method keep? Think about Notebook 2: our masks skip **rows** of k-space. Which direction of fine detail does that affect?

In [ ]:
base = truth[test][3]
features = base.copy()
features[60:64, 50:54] += 0.35          # small bright square
features[70:90, 40] += 0.35             # thin vertical line
features[75, 70:90] += 0.35             # thin horizontal line
features = np.clip(features, 0, 1)

def feature_contrast(img):
    # how much brighter each feature is than the pixels right next to it
    square = img[60:64, 50:54].mean() - img[66:70, 50:54].mean()
    vertical = img[72:88, 40].mean() - 0.5 * (img[72:88, 38].mean() + img[72:88, 42].mean())
    horizontal = img[75, 72:88].mean() - 0.5 * (img[73, 72:88].mean() + img[77, 72:88].mean())
    return square, vertical, horizontal

recons = reconstruct_all(features, lines)
print(f"{'':<20}{'square':>8}{'vertical':>10}{'horizontal':>12}   (feature contrast)")
print(f"{'ground truth':<20}" + "".join(f"{v:>{w}.2f}" for v, w in zip(feature_contrast(features), [8, 10, 12])))
for m in METHODS:
    print(f"{m:<20}" + "".join(f"{v:>{w}.2f}" for v, w in zip(feature_contrast(recons[m]), [8, 10, 12])))

zoom = (slice(35, 95), slice(30, 95))
show([features[zoom]] + [recons[m][zoom] for m in METHODS], ["ground truth"] + METHODS, vmin=0, vmax=0.7)

**Interpretation:** the three features tell three different stories.

- **The vertical line is kept by every method.** It is thin in the left-right direction, so it needs high $k_x$ frequencies. Every measured row contains all of $k_x$, so this detail was fully measured.
- **The horizontal line is lost by every method**, even compressed sensing and the unrolled network. It is thin in the up-down direction, so it needs high $k_y$ frequencies, which are exactly the rows our mask mostly skips (Notebook 2). This is a limit of the **sampling**, not of any method: what was not measured and cannot be predicted from the rest of the image is gone.
- **The small square separates the methods.** The U-Net keeps only about a quarter of its contrast, *less than zero-filling*: it has learned to remove things that do not look like its training images, and it never saw a square. The unrolled network and compressed sensing keep most of it, because data consistency puts back the measured k-space lines that carry the square.

Two lessons: test with features that were **not** in the training data, and when something is lost, check whether the sampling or the method lost it.

## 5. When the scan changes

A network is trained for one setting, but scanners are used in many. We test all methods, with no retraining, under four changes:

- a **different random mask** with the same R = 4 (another protocol, or another scanner);
- a **higher acceleration**, R = 6;
- **noise** in k-space, at two levels (our training data had none).

> 🤔 **Predict:** Which methods will suffer most? Remember that compressed sensing has no training data at all.

In [ ]:
conditions = {
    "training setup": dict(lines=lines),
    "new mask, R = 4": dict(lines=variable_density_lines(N, 4, seed=7)),
    "R = 6": dict(lines=variable_density_lines(N, 6)),
    "noise 0.02": dict(lines=lines, noise=0.02),
    "noise 0.05": dict(lines=lines, noise=0.05),
}
n_shift = 10
shift_errors = {m: [] for m in METHODS}
for name, settings in conditions.items():
    errors = {m: [] for m in METHODS}
    for i in range(n_shift):
        recons = reconstruct_all(truth[test][i], seed=i, **settings)
        for m in METHODS:
            errors[m].append(nrmse(recons[m], truth[test][i]))
    for m in METHODS:
        shift_errors[m].append(np.mean(errors[m]))

print(f"{'condition':<18}" + "".join(f"{m:>20}" for m in METHODS))
for j, name in enumerate(conditions):
    print(f"{name:<18}" + "".join(f"{shift_errors[m][j]:>20.3f}" for m in METHODS))

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.2
positions = np.arange(len(conditions))
for j, (m, color) in enumerate(zip(METHODS, COLORS)):
    ax.bar(positions + (j - 1.5) * width, shift_errors[m], width * 0.92, color=color, label=m)
ax.set_xticks(positions)
ax.set_xticklabels(list(conditions))
ax.set_ylabel(f"mean NRMSE, {n_shift} test phantoms")
ax.set_title("Every method gets worse when the scan changes", loc="left")
ax.legend(frameon=False, ncol=4, loc="upper left")
ax.set_ylim(0, max(max(v) for v in shift_errors.values()) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
plt.tight_layout()
plt.show()

**Interpretation:**

- **Every method** degrades under every change; the scan itself carries less, or noisier, information.
- On these phantoms, the **unrolled network stays the most accurate** in every condition. Its data consistency step uses the actual mask of each scan, so even with a new mask its error stays far below the U-Net's, which only ever saw one artifact pattern.
- But the networks lose much of their **advantage**. With a new mask, the unrolled network's error rises by about 40%; at R = 6 it is almost five times its training value and much closer to compressed sensing. Noise roughly doubles it.

And these test images are still phantoms just like the training data. Section 6 removes that last advantage.

## 6. Images unlike the training data

The strongest test is an image that is simply *different*. We try two:

1. A **head without a skull ring**: one large ellipse with three internal structures, including a small bright dot. Every training phantom had a bright skull.
2. A **photograph** (the classic "cameraman" test image), with textures, gradients and no black background. It is not an MR image, but it stands in for "anatomy the network has never seen".

> 🤔 **Predict:** Will the networks still beat zero-filling on these images? Will they invent anything that is not there?

In [ ]:
# 1. a head without a skull, drawn with the same ellipse method as the training phantoms
M = 2 * N
yy, xx = np.mgrid[-1:1:M * 1j, -1:1:M * 1j]
fine = np.zeros((M, M))
for cx, cy, a, b, angle, value in [(0, 0, 0.62, 0.82, 0.1, 0.25),          # brain, no skull
                                   (-0.2, 0.1, 0.15, 0.25, 0.4, 0.15),     # bright structure
                                   (0.25, -0.2, 0.12, 0.12, 0, -0.12),     # dark structure
                                   (0.05, 0.4, 0.03, 0.03, 0, 0.2)]:       # small bright dot
    c, s = np.cos(angle), np.sin(angle)
    xr, yr = (xx - cx) * c + (yy - cy) * s, -(xx - cx) * s + (yy - cy) * c
    fine[(xr / a) ** 2 + (yr / b) ** 2 <= 1] += value
no_skull = np.clip(fine, 0, 1).reshape(N, 2, N, 2).mean(axis=(1, 3))

# 2. a photograph
photo = resize(camera() / 255.0, (N, N), anti_aliasing=True)

for title, image in [("head without skull", no_skull), ("photograph", photo)]:
    recons = reconstruct_all(image, lines)
    print(f"{title}:")
    for m in METHODS:
        print(f"  {m:<20} NRMSE {nrmse(recons[m], image):.3f}   SSIM {ssim(recons[m], image):.3f}")
    show([image] + [recons[m] for m in METHODS], ["ground truth"] + METHODS, vmin=0, vmax=1)
    show([recons[m] - image for m in METHODS], [f"error: {m}" for m in METHODS], cmap="RdBu_r", vmin=-0.3, vmax=0.3)

Look at the top and bottom edges of the photograph in the error maps. Compare the average error size there with the rest of the image:

In [ ]:
recons = reconstruct_all(photo, lines)
edge_rows, inner_rows = np.r_[0:4, N - 4:N], np.r_[4:N - 4]
print(f"{'method':<20}{'edge rows':>10}{'rest':>8}{'ratio':>8}   (mean absolute error)")
for m in METHODS:
    err = np.abs(recons[m] - photo)
    print(f"{m:<20}{err[edge_rows].mean():>10.3f}{err[inner_rows].mean():>8.3f}{err[edge_rows].mean() / err[inner_rows].mean():>8.1f}")

**Interpretation:**

- **Head without a skull.** The U-Net does *worse than zero-filling*: its error map shows that it darkens the whole head and blurs away the small bright dot. The unrolled network and compressed sensing are about equally good. Note what did *not* happen: no network invented a skull, even though every training image had one. A network does not always hallucinate, but you only know by testing.
- **Photograph.** **Both networks do worse than zero-filling**, while compressed sensing, which was never trained on anything, is the best method. The unrolled network adds horizontal stripes, and its errors in the top and bottom rows are about five times larger than in the rest of the image: every training phantom had a black border, and the network learned to expect one.

This is the most important lesson of the module. A network learns the **training data**, not "MRI". In practice, the equivalent of our photograph is a scanner, a field strength, a contrast, a body part, a patient group or a disease that was missing from training. Hand-made assumptions like sparsity are weaker but more general; learned ones are stronger but only within their training distribution.

## 7. A checklist for evaluating reconstruction methods

Everything in this module points to the same questions. Use them when you read a paper, review a product claim, or evaluate your own method:

1. **Fair baselines.** Were the comparison methods tuned as carefully as the new one (λ for compressed sensing, training time for networks)? *(Notebook 4, Exercise 1)*
2. **Separate test data.** Were the test images, and any parameter choices, kept apart from training? Was the mask or λ chosen on the test images? *(Notebooks 2 and 3)*
3. **More than one metric.** Are NRMSE/PSNR *and* SSIM reported, and do they agree? *(Sections 2–3)*
4. **Small structures and task-based tests.** Were small lesions or other clinically relevant features measured directly, not only whole-image averages? *(Section 4)*
5. **Several masks and random seeds.** Do results hold for more than one sampling pattern? *(Notebook 2, Exercise 3; Section 5)*
6. **Distribution shift.** Was the method tested on other scanners, noise levels, accelerations, body parts or patient groups than it was trained on? *(Sections 5–6)*
7. **Failure cases.** Does the paper show where the method fails, not only its best images?
8. **Readers.** For clinical use: did radiologists, blinded to the method, rate the images for diagnosis? No pixel metric replaces this.

These questions are an active research area. The 2020 fastMRI challenge, for example, found that at high acceleration even top-ranked methods sometimes lost small abnormalities that radiologists noticed, despite excellent SSIM (Muckley et al., 2021).

## Exercises

Solutions are in `solutions/05_solutions.ipynb`.

### Exercise 1: where does SSIM disagree?

`structural_similarity(reference, recon, data_range=1.0, full=True)` also returns an **SSIM map**: one value per pixel. Compute the maps for compressed sensing and the U-Net on test phantom 0, and show them side by side. Where in the image does each method lose structural similarity? Does this explain why SSIM ranks them differently from NRMSE?

In [ ]:
# Exercise 1: your code here


### Exercise 2: how small is too small?

Add bright squares of size 1, 2, 3, 4 and 6 pixels (one at a time, at `features[60:60+s, 50:50+s]`, brightness +0.35) to test phantom 3, and measure the contrast each method keeps, as a fraction of the true contrast. Plot the fraction against the square size, one line per method. Which method keeps small features best?

<details><summary>Hint</summary>

Adapt <code>feature_contrast</code>: compare the mean inside the square with the mean of a same-sized patch just below it. Divide by the same number for the ground truth.
</details>

In [ ]:
# Exercise 2: your code here


### Exercise 3: one mask is not enough

Repeat the "new mask, R = 4" test of Section 5 with **five** different mask seeds (1 to 5), each on 10 test phantoms. For each method, report the mean NRMSE and the range (best and worst seed). Which method is most sensitive to the mask?

In [ ]:
# Exercise 3: your code here


### Exercise 4: review a claim

A (fictional) paper abstract says:

> *"Our deep network reconstructs 8× accelerated brain MRI with an SSIM of 0.97, outperforming compressed sensing (SSIM 0.89). It can therefore replace full-length scans in clinical practice."*

Using the checklist in Section 7, write down at least five questions you would ask the authors before believing the final sentence. (There is no code for this exercise.)

*Your answer here.*

## Summary

- **NRMSE and PSNR** measure total squared error; **SSIM** measures local structure. They can rank methods differently, and none of them measures diagnostic value directly.
- **Small, unfamiliar features** can be lost. Some losses come from the **sampling** (fine detail along the undersampled direction is simply not measured); others come from the **method** (a network removing what it never saw in training).
- **Data consistency** helps a network keep what was measured, under new masks and on unfamiliar images.
- Every method degrades when the **scan changes**, and a network can do **worse than no reconstruction at all** on images unlike its training data.
- Evaluate reconstruction methods with fair baselines, several metrics, task-based tests, several masks, distribution shift and, ultimately, radiologists.

You have now gone from the Fourier transform to deep learning reconstruction, and learned how to judge it critically. **Next:** Notebook 6 applies the same methods to real scanner data from the M4Raw dataset.

## References

1. Wang, Z., Bovik, A. C., Sheikh, H. R., & Simoncelli, E. P. (2004). Image quality assessment: From error visibility to structural similarity. *IEEE Transactions on Image Processing*, 13(4), 600–612.
2. Muckley, M. J., et al. (2021). Results of the 2020 fastMRI challenge for machine learning MR image reconstruction. *IEEE Transactions on Medical Imaging*, 40(9), 2306–2317.
3. Antun, V., Renna, F., Poon, C., Adcock, B., & Hansen, A. C. (2020). On instabilities of deep learning in image reconstruction and the potential costs of AI. *Proceedings of the National Academy of Sciences*, 117(48), 30088–30095.
4. Knoll, F., et al. (2019). Assessment of the generalization of learned image reconstruction and the potential for transfer learning. *Magnetic Resonance in Medicine*, 81(1), 116–128.

---
*Text and figures: CC-BY 4.0. Code: MIT License.*